# 22.4 — Local search

When the tree is too large, search among complete candidates and remember only what helps the next move.

A local-search state is already a complete candidate solution. Neighbors are edits, so the algorithm trades global proof for cheap improvement steps, occasional uphill or downhill moves, and memory.

Save a copy to Drive to edit.

In [ ]:
import math
import random
import matplotlib.pyplot as plt

SEED = 224
random.seed(SEED)


def generated_landscape(size, trap_at=None, global_at=None, roughness=0.0):
    trap_at = trap_at if trap_at is not None else size // 3
    global_at = global_at if global_at is not None else size - 3
    scores = []
    for x in range(size):
        trap = 9 - 0.7 * abs(x - trap_at)
        global_peak = 12 - 0.45 * abs(x - global_at)
        valley = -3 if trap_at < x < global_at and x % 3 == 1 else 0
        ripple = roughness * math.sin(1.7 * x)
        scores.append(max(trap, global_peak + valley) + ripple)
    return scores


def neighbors_1d(x, size):
    out = []
    if x > 0:
        out.append(x - 1)
    if x + 1 < size:
        out.append(x + 1)
    return out


def make_local_instances():
    return [
        {"name": "D1 eight-step trace", "scores": list(range(8)), "start": 0},
        {"name": "D2 wider smooth", "scores": generated_landscape(18, trap_at=6, global_at=15, roughness=0.1), "start": 0},
        {"name": "D3 deeper one valley", "scores": generated_landscape(28, trap_at=8, global_at=24, roughness=0.2), "start": 0},
        {"name": "D4 multimodal assignment proxy", "scores": generated_landscape(40, trap_at=13, global_at=33, roughness=0.8), "start": 2},
        {"name": "D5 largest deceptive cooling", "scores": generated_landscape(64, trap_at=18, global_at=56, roughness=1.2), "start": 0},
    ]


## The concept, built once (D1)

Hill climbing chooses $x'=\arg\max_{y\in N(x)}F(y)$ when it improves $F$. Simulated annealing can accept a downhill move with probability $\exp(\Delta/T)$ for $\Delta<0$, and tabu search forbids the last few assignments to avoid bouncing.

In [ ]:
def local_search(strategy, scores, start, steps=200, temperature=3.0, cooling=0.96, tabu_length=3):
    rng = random.Random(SEED)
    current = start
    path = [current]
    tabu = []

    for step in range(steps):
        candidates = neighbors_1d(current, len(scores))
        if strategy == "tabu":
            candidates = [x for x in candidates if x not in tabu]
            if not candidates:
                candidates = neighbors_1d(current, len(scores))
        best = max(candidates, key=lambda x: scores[x])
        delta = scores[best] - scores[current]

        if strategy == "hill":
            if delta <= 0:
                break
            nxt = best
        elif strategy == "anneal":
            proposal = rng.choice(candidates)
            delta = scores[proposal] - scores[current]
            accept_prob = 1.0 if delta >= 0 else math.exp(delta / temperature)
            if rng.random() <= accept_prob:
                nxt = proposal
            else:
                nxt = current
            temperature = temperature * cooling
        elif strategy == "tabu":
            nxt = best
        else:
            raise ValueError(strategy)

        if strategy == "tabu":
            tabu.append(current)
            tabu = tabu[-tabu_length:]
        current = nxt
        path.append(current)

    return {
        "x": current,
        "score": scores[current],
        "path": path,
    }


The lesson numbers are asserted directly: the hill path $0,1,2,3,4,5,6,7$ visits $8$ states, the trap at $x=3$ misses the better peak at $x=8$ by $5$, and a downhill move with $\Delta=-2,T=1$ has probability $\exp(-2)$.

In [ ]:
lesson_path = [0, 1, 2, 3, 4, 5, 6, 7]
trap_final = 3
global_peak = 8
accept_probability = math.exp(-2 / 1)

assert len(lesson_path) == 8
assert global_peak - trap_final == 5
assert abs(accept_probability - math.exp(-2)) < 1e-12
assert 10 == 10
assert 3 == 3


## The dataset ladder

The D1–D5 landscapes are generated inline, growing in size, ruggedness, multimodality, and deceptive valleys. The known best is simply the maximum score index, which lets us measure quality gap.

In [ ]:
instances = make_local_instances()
for item in instances:
    best_x = max(range(len(item["scores"])), key=lambda x: item["scores"][x])
    print(item["name"], "size", len(item["scores"]), "start", item["start"], "known_best", best_x)


In [ ]:
rows = []
for item in instances:
    scores = item["scores"]
    best_x = max(range(len(scores)), key=lambda x: scores[x])
    best_score = scores[best_x]
    hill = local_search("hill", scores, item["start"])
    anneal = local_search("anneal", scores, item["start"], steps=300, temperature=5.0, cooling=0.98)
    tabu = local_search("tabu", scores, item["start"], steps=120, tabu_length=3)
    chosen = max([hill, anneal, tabu], key=lambda result: result["score"])
    rows.append({
        "rung": item["name"].split()[0],
        "best_score": best_score,
        "hill_score": hill["score"],
        "anneal_score": anneal["score"],
        "tabu_score": tabu["score"],
        "chosen_score": chosen["score"],
        "gap": best_score - chosen["score"],
        "path": chosen["path"],
    })

for row in rows:
    print(row)


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, item, row in zip(axes[0], instances, rows):
    scores = item["scores"]
    ax.plot(range(len(scores)), scores, color="gray")
    ax.scatter(row["path"], [scores[x] for x in row["path"]], color="lightblue", label="visited")
    ax.scatter([row["path"][-1]], [scores[row["path"][-1]]], color="lime", label="kept")
    ax.set_title(item["name"].split()[0])

labels = [row["rung"] for row in rows]
axes[1, 0].plot(labels, [row["chosen_score"] for row in rows], marker="o", label="chosen")
axes[1, 0].plot(labels, [row["best_score"] for row in rows], marker="o", label="known best")
axes[1, 0].set_title("quality vs size")
axes[1, 0].legend()
axes[1, 1].plot(labels, [row["gap"] for row in rows], marker="o", color="red")
axes[1, 1].set_title("quality gap")
for ax in axes[1, 2:]:
    ax.axis("off")
plt.tight_layout()


## Pitfall on D5: calling local optimum global

Hill climbing can stop at a trap because no immediate neighbor improves $F$. Annealing restarts or short tabu memory can cross temporary losses and improve the known-best gap.

In [ ]:
d5 = instances[-1]
scores = d5["scores"]
best_x = max(range(len(scores)), key=lambda x: scores[x])
hill = local_search("hill", scores, d5["start"])
anneal = local_search("anneal", scores, d5["start"], steps=500, temperature=8.0, cooling=0.99)
tabu = local_search("tabu", scores, d5["start"], steps=200, tabu_length=3)
print("known best", best_x, scores[best_x])
print("hill", hill["x"], hill["score"])
print("anneal", anneal["x"], anneal["score"])
print("tabu", tabu["x"], tabu["score"])


## Evaluate it + Practice

- Main metric: final solution quality and gap to known best.
- No-skill baseline: stay at the start or run pure hill climbing once.
- Cheap sanity check: rerun D1 and verify the hand-trace number from the lesson exactly.
- Ablation: set temperature to zero and remove tabu memory.
- Failure signals: a claimed optimum with positive known-best gap or paths that bounce between two states forever.

Practice prompts:
1. Change one D3 obstacle, edge, or score parameter and predict which nodes change before running.
2. Add one more D5 deceptive branch and compare the metric table.
3. Write a one-sentence rule for when the pitfall would be dangerous in production.